# Guide-1_极简实验流程


In [ ]:
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "data" / "loader_roots").exists():
            return path
    raise FileNotFoundError("Could not find data/loader_roots from the current working directory.")


PROJECT_ROOT = find_project_root()
XJTU_ROOT = str(PROJECT_ROOT / "data" / "loader_roots" / "xjtu")
PHM2012_ROOT = str(PROJECT_ROOT / "data" / "loader_roots" / "phm2012")

from phm.data.labeler.BearingRulLabeler import BearingRulLabeler
from phm.data.loader.XJTULoader import XJTULoader
from phm.engine.Evaluator import Evaluator
from phm.engine.metric.MSE import MSE
from phm.engine.metric.MAE import MAE
from phm.engine.metric.PHM2012Score import PHM2012Score
from phm.engine.metric.PercentError import PercentError
from phm.engine.metric.RMSE import RMSE
from phm.engine.tester.BaseTester import BaseTester
from phm.engine.trainer.BaseTrainer import BaseTrainer
from phm.model.basic.CNN import CNN

# Step 1: Load raw data
data_loader = XJTULoader(XJTU_ROOT)
bearing = data_loader.load_entity('Bearing1_1')

# Step 2: Construct dataset
labeler = BearingRulLabeler(2048)
dataset = labeler.label(bearing, 'Horizontal Vibration')
train_set, test_set = dataset.split_by_ratio(0.7)

# Step 3: Train model
model = CNN(input_size=2048, output_size=1)
trainer = BaseTrainer()
trainer.train(model, train_set)

# Step 4: Test model
tester = BaseTester()
result = tester.test(model, test_set)

# Step 5: Evaluate results
evaluator = Evaluator()
evaluator.add(MAE(), MSE(), RMSE(), PercentError(), PHM2012Score())
evaluator(test_set, result)
